# Transfer across two simulated packages

Train a six-control model on package A and see how many examples it takes to adapt to B.
This example uses six independent rings in the 1550 nm band. The dual-ring notebook uses the coupled-ring model.

In [ ]:
import copy
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch as pt
import torch.nn as nn
import torch.optim as optim


device = pt.device("cuda" if pt.cuda.is_available() else "cpu")
print(device)

## Generate the data

The packages have different thermal coupling, routing resistance and ambient temperature.
The model predicts squared voltage scaled to 0–1. Convert back to volts for scoring.

In [ ]:
N_POINTS = 5001
LAMBDA_START = 1540e-9
LAMBDA_STOP = 1560e-9
LAMBDA_0 = 1550e-9

N_EFF = 2.4
N_G = 4.2
DNEFF_DT = 1.8e-4
LOSS_DB_PER_CM = 3.0
RING_RADIUS = 5e-6
SELF_COUPLING = 0.995
HEATER_RESISTANCE = 1000.0
V_MAX = 3.0
NOISE_DB = 0.05
FLOOR_DB = -40.0

RING_LENGTH = 2 * np.pi * RING_RADIUS
ROUND_TRIP_AMPLITUDE = np.exp(-LOSS_DB_PER_CM * 100 * np.log(10) / 20 * RING_LENGTH)


def ring_phase(wavelengths, delta_t, phase_offset):
    """Round-trip phase of one ring, shape (samples, points)."""
    # First-order dispersion so the FSR comes out set by N_G, plus the thermo-optic shift.
    n_eff = N_EFF - (N_G - N_EFF) * (wavelengths - LAMBDA_0) / LAMBDA_0 + DNEFF_DT * delta_t[:, None]
    return 2 * np.pi * n_eff * RING_LENGTH / wavelengths + phase_offset


def bus_spectrum_db(wavelengths, ring_delta_t, phase_offsets):
    """Bus transmission in dB for ring temperature rises of shape (samples, rings)."""
    r, a = SELF_COUPLING, ROUND_TRIP_AMPLITUDE
    field = np.ones((ring_delta_t.shape[0], wavelengths.size), dtype=complex)
    for ring in range(ring_delta_t.shape[1]):
        e = np.exp(1j * ring_phase(wavelengths, ring_delta_t[:, ring], phase_offsets[ring]))
        field *= (r - a * e) / (1 - r * a * e)
    return 10 * np.log10(np.maximum(np.abs(field) ** 2, 10 ** (FLOOR_DB / 10)))


N_HEATERS = 6
PITCH_UM = 60
ALPHA = 0.003
WAVELENGTHS = np.linspace(LAMBDA_START, LAMBDA_STOP, N_POINTS)
PM_PER_K = LAMBDA_0 * DNEFF_DT / N_G * 1e12  # resonance shift per K
PHASE_OFFSETS = 2 * np.pi * (0.15 + np.arange(N_HEATERS) / N_HEATERS)  # heaters-off resonances, evenly spaced over one FSR


def coupling(k_self, spread_um):
    x = np.arange(N_HEATERS) * PITCH_UM
    return k_self * np.exp(-np.abs(x[:, None] - x[None, :]) / spread_um)


PACKAGES = {
    "A": dict(coupling=coupling(2.4, 40), series_ohm=0.0, ambient_k=0.0),
    "B": dict(coupling=coupling(1.8, 75), series_ohm=60.0, ambient_k=1.5),
}


def ring_temps(pkg, v):
    # ring temperature rise (K) for heater voltages v
    r = HEATER_RESISTANCE
    p_mw = v**2 * r / (r + pkg["series_ohm"]) ** 2 * 1e3
    dt = p_mw @ pkg["coupling"].T
    return pkg["ambient_k"] + dt * (1 + ALPHA * dt)


def simulate(pkg, n, rng):
    # Draw uniformly in power, then convert to volts.
    v = V_MAX * np.sqrt(rng.uniform(size=(n, N_HEATERS)))
    dt = ring_temps(pkg, v)
    chunks = np.array_split(dt, max(1, n // 500))
    spectra = np.vstack([bus_spectrum_db(WAVELENGTHS, c, PHASE_OFFSETS) for c in chunks])
    spectra += rng.normal(0, NOISE_DB, spectra.shape)
    return spectra.astype(np.float32), v.astype(np.float32)

In [ ]:
rng = np.random.default_rng(0)
n_train = 4000
S_A, V_A = simulate(PACKAGES["A"], 5000, rng)
S_pool, V_pool = simulate(PACKAGES["B"], 1000, rng)
S_test, V_test = simulate(PACKAGES["B"], 1000, rng)
S_B_full, V_B_full = simulate(PACKAGES["B"], n_train, rng)

X_MEAN, X_STD = S_A[:n_train].mean(), S_A[:n_train].std()


def to_x(S):
    return pt.tensor((S - X_MEAN) / X_STD).unsqueeze(1).to(device)


def to_u(V):
    return pt.tensor((V / V_MAX) ** 2).to(device)


def u_to_v(u):
    return V_MAX * np.sqrt(np.clip(u, 0, 1))


X_A, U_A = to_x(S_A), to_u(V_A)
X_pool, U_pool = to_x(S_pool), to_u(V_pool)
X_test = to_x(S_test)
X_B_full, U_B_full = to_x(S_B_full), to_u(V_B_full)

## Train the models

Use 4,000 spectra for each full training run. Package B also has a separate calibration pool and test set.

In [ ]:
class NiuNeuralNetwork(nn.Module):
    def __init__(self, input_size, num_outputs):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=6)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=6)
        self.pool2 = nn.MaxPool1d(2)
        self.relu = nn.ReLU()

        flat_size = self._flat_size(input_size)
        self.fc1 = nn.Linear(flat_size, 128)
        self.fc2 = nn.Linear(128, num_outputs)

    def _flat_size(self, input_size):
        x = pt.zeros(1, 1, input_size)
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        return x.numel()

    def hidden(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.flatten(1)
        return self.relu(self.fc1(x))

    def forward(self, x):
        return self.fc2(self.hidden(x))


def fit(model, X, Y, steps, lr, warmup=0, cosine=False, clip=None, seed=0, batch_size=32):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr)

    # linear warmup, then cosine decay down to lr/100
    def lr_scale(step):
        scale = min(1.0, (step + 1) / warmup) if warmup else 1.0
        if cosine:
            scale *= 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * step / steps))
        return scale

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
    gen = pt.Generator().manual_seed(seed)
    model.train()
    for _ in range(steps):
        idx = pt.randint(len(X), (min(batch_size, len(X)),), generator=gen).to(device)
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(model(X[idx]), Y[idx])
        loss.backward()
        if clip:
            nn.utils.clip_grad_norm_(params, clip)
        optimizer.step()
        scheduler.step()
    return model


def predict(model, X):
    model.eval()
    with pt.no_grad():
        return pt.cat([model(X[i:i + 500]).cpu() for i in range(0, len(X), 500)]).numpy()


def train_full(X, Y, seed=0):
    pt.manual_seed(seed)
    model = NiuNeuralNetwork(N_POINTS, N_HEATERS).to(device)
    return fit(model, X, Y, steps=5000, lr=1e-3, warmup=500, cosine=True, clip=1.0, seed=seed)

source = train_full(X_A[:n_train], U_A[:n_train])

In [ ]:
def score(pkg, v_pred, v_true):
    err_pm = (ring_temps(pkg, v_pred) - ring_temps(pkg, v_true)) * PM_PER_K
    per_spectrum = np.sqrt(np.mean(err_pm**2, axis=1))
    return dict(resonance_pm=np.sqrt(np.mean(err_pm**2)), median_pm=np.median(per_spectrum),
                voltage_mae_mv=1e3 * np.mean(np.abs(v_pred - v_true)))


reference = train_full(X_B_full, U_B_full)

U_src_pool = predict(source, X_pool)
U_src_test = predict(source, X_test)

baseline = pd.DataFrame({
    "source on A": score(PACKAGES["A"], u_to_v(predict(source, X_A[n_train:])), V_A[n_train:]),
    "source on B, no adaptation": score(PACKAGES["B"], u_to_v(U_src_test), V_test),
    "reference (4000 B spectra)": score(PACKAGES["B"], u_to_v(predict(reference, X_test)), V_test),
}).T
baseline.round(1)

## Calibrate with fewer spectra

Fit an affine map to the source model's power predictions and compare it with fine-tuning and training from scratch.
The sweep takes longer than the two full fits.

In [ ]:
def ridge_toward(A, Y, W0, lam):
    # least squares fit, pulled slightly toward W0 so it still works with fewer than 7 spectra
    A = np.hstack([A, np.ones((len(A), 1))])
    return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ Y + lam * W0)


W_IDENTITY = np.vstack([np.eye(N_HEATERS), np.zeros((1, N_HEATERS))])  # no correction


def thermal_adapter(idx, _seed):
    W = ridge_toward(U_src_pool[idx], U_pool[idx].cpu().numpy(), W_IDENTITY, lam=1e-3)
    return U_src_test @ W[:-1] + W[-1]


def finetune_all(idx, seed):
    model = copy.deepcopy(source)
    fit(model, X_pool[idx], U_pool[idx], steps=200, lr=1e-4, seed=seed)
    return predict(model, X_test)


def from_scratch(idx, seed):
    pt.manual_seed(seed)
    model = NiuNeuralNetwork(N_POINTS, N_HEATERS).to(device)
    fit(model, X_pool[idx], U_pool[idx], steps=2000, lr=1e-3, warmup=500, cosine=True, clip=1.0, seed=seed)
    return predict(model, X_test)


methods = {"thermal adapter": thermal_adapter, "fine-tune all": finetune_all, "train from scratch": from_scratch}
N_TARGET = [5, 10, 20, 50, 100, 200, 500, 1000]

rows = []
for method, run in methods.items():
    for n in N_TARGET:
        for seed in range(3):
            idx = np.random.default_rng(100 + seed).choice(len(X_pool), n, replace=False)
            u_pred = run(idx, seed)
            rows.append(dict(method=method, n=n, seed=seed, **score(PACKAGES["B"], u_to_v(u_pred), V_test)))
    print(method, "done")

results = pd.DataFrame(rows)
summary = results.groupby(["method", "n"], sort=False)["resonance_pm"].mean().unstack("n")
summary.round(1)

In [ ]:
for method, row in summary.iterrows():
    plt.plot(row.index, row.values, marker="o", label=method)
plt.axhline(baseline.loc["reference (4000 B spectra)", "resonance_pm"], color="gray", linestyle="--", label="reference (4000 B spectra)")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("labelled B spectra (N)")
plt.ylabel("resonance error, RMS (pm)")
plt.legend()
plt.show()

Compare the fitted power map with the one calculated from the simulated thermal matrices.

In [ ]:
def drive_gain(pkg):
    # mW per unit of u
    r = HEATER_RESISTANCE
    return 1e3 * V_MAX**2 * r / (r + pkg["series_ohm"]) ** 2


A, B = PACKAGES["A"], PACKAGES["B"]
M_physics = drive_gain(A) / drive_gain(B) * np.linalg.solve(B["coupling"], A["coupling"])
b_physics = np.linalg.solve(B["coupling"], np.full(N_HEATERS, A["ambient_k"] - B["ambient_k"])) / drive_gain(B)

idx = np.random.default_rng(100).choice(len(X_pool), 50, replace=False)
W = ridge_toward(U_src_pool[idx], U_pool[idx].cpu().numpy(), W_IDENTITY, lam=1e-3)

print("M from physics:\n", M_physics)
print("M fitted:\n", W[:-1].T)
print("b from physics:", b_physics)
print("b fitted:      ", W[-1])